In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score

import joblib

In [ ]:
telco_df = pd.read_csv('datasets/telco.csv')

In [ ]:
telco_df.shape

In [ ]:
telco_df.head()

In [ ]:
telco_df['City'].nunique()

In [ ]:
list(telco_df.columns)

In [ ]:
print(telco_df['Churn Label'].unique())
print(telco_df['Churn Label'].value_counts())

In [ ]:
telco_df.info()

In [ ]:
telco_df.describe()

In [ ]:
telco_clean = telco_df.drop(columns=[
    'Customer ID',
    'Country',
    'State',
    'City',
    'Zip Code',
    'Churn Category',
    'Churn Reason',
    'Churn Score'
])

In [ ]:
telco_clean.head()

In [ ]:
telco_clean['Customer Status'].unique()

In [ ]:
telco_clean[telco_clean['Customer Status'] == 'Churned']

In [ ]:
pd.crosstab(telco_clean['Married'], telco_clean['Churn Label'])

In [ ]:
telco_clean['Gender'].value_counts()

In [ ]:
telco_clean['Gender'].value_counts().plot(kind='bar')

plt.title('Gender Distrubution')
plt.xlabel('Gender')
plt.ylabel('Number of Customers')
plt.show()

In [ ]:
pd.crosstab(telco_clean['Gender'], telco_clean['Churn Label'])

In [ ]:
telco_clean.isna().sum()

In [ ]:
telco_clean.head()

In [ ]:
telco_clean['Churn Label'].value_counts(normalize=True)*100

In [ ]:
telco_clean['Churn Label'].value_counts()

In [ ]:
telco_clean.isnull().sum().sort_values(ascending=False)

In [ ]:
pd.crosstab(
    telco_clean['Internet Service'],
    telco_clean['Internet Type'],
    dropna=False
)

In [ ]:
pd.crosstab(
    telco_clean['Internet Service'],
    telco_clean['Offer'],
    dropna=False
)

In [ ]:
telco_clean['Internet Type'] = telco_clean['Internet Type'].fillna('No Internet')
telco_clean['Offer'] = telco_clean['Offer'].fillna('No Offer')

In [ ]:
telco_clean.duplicated().sum()

In [ ]:
X = telco_clean.drop(columns=['Churn Label'])
y = telco_clean['Churn Label']

In [ ]:
telco_clean.head()

In [ ]:
X = X.drop(columns=['Customer Status'])

In [ ]:
print(X.shape)
print(y.shape)

In [ ]:
X.columns.tolist()

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

In [ ]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

In [ ]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

print(y_train.value_counts(normalize=True))
print(y_val.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

In [ ]:
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['string']).columns.tolist()

In [ ]:
numeric_cols

In [ ]:
categorical_cols

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

In [ ]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

In [ ]:
model.fit(X_train, y_train)

In [ ]:
val_predictions = model.predict(X_val)

print('Validation accuracy: ', accuracy_score(y_val, val_predictions))

In [ ]:
print(classification_report(y_val, val_predictions))

In [ ]:
cm = confusion_matrix(y_val, val_predictions)

print(cm)

ConfusionMatrixDisplay(
    confusion_matrix=cm, 
    display_labels=model.classes_
).plot()

plt.show()

In [ ]:
joblib.dump(model, "../model/churn_model.pkl")

In [ ]:
X_train.columns.tolist()